[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordADH2026PythonBootcamp/blob/main/solutions/10_medvision_lab.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util, subprocess, sys

if importlib.util.find_spec("adh_python") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/BradSegal/OxfordADH2026PythonBootcamp.git"],
        check=True,
    )
print("Course helpers ready.")

# MedVision lab: testing an AI model on chest X-rays

In this lab you'll interact with **MedGemma 1.5 4B**, a vision language
model from Google that has been trained on medical images and text. You will
show it chest X-rays, ask it questions, iteratively modify the questions,
attempt to break the model, and finally run a small evaluation you put
together.

The goal is not to learn how to use this model clinically, but how to
interrogate any AI tool that you are given, what to ask, how to catch it
out, and what evidence you would want to see to trust it to handle a
patient.

**By the end of this lab you can:**
- run a vision-language model on an X-ray and compare its reply with a
  reference report;
- change a prompt one thing at a time and see what each change does;
- recognise invented findings, leading questions and automation bias; and
- record a small evaluation as a list of dicts and say what it can and
  cannot show.

**Before you start, select a GPU.** A GPU (graphics processing unit) is the
type of chip that enables AI models to run quickly. In Colab select
**Runtime > Change runtime type > T4 GPU > Save**. The free T4 is
sufficient.

**If you do not have a GPU, you can work through everything except trying
your own prompts.** If you do not have access to a GPU you can still run all
the cells, however, whenever the model is asked to respond, it shows you a
pre-recorded response we saved when running this course. This is clearly
marked as a saved response. These were recorded using a compressed version
of the same model running on a normal computer and so differ from what the
model would respond with if you were to run it live. If you want to try your
own prompts, this is not possible as there is not a saved response to use
and so is optional and requires a GPU. The lab provides an alternative that
leverages the saved responses.

**No account or token needed.** The model is downloaded from an open copy
on Hugging Face (`unsloth/medgemma-1.5-4b-it`), so you do not need a
Hugging Face account, a token or an access request. Its use is still
governed by Google's Health AI Developer Foundations terms of use, and its
model card says its outputs are not intended to inform diagnosis or
treatment directly.

The lab takes ± 75 mins. The first download may take 5-10 mins.

## Why this matters

AI tools that can read a chest X-ray are already being used in parts of the
NHS, largely to flag a film to be reviewed sooner by a radiologist. These
tools are regulated as medical devices, have a narrow role, and have been
evaluated for their narrow task. Despite this, NICE still considers the
evidence limited: in their early value assessment of these tools for
suspected lung cancer (HTE12, 2023, updated 2025) they state that these
tools should only be used in a research context or while more evidence is
gathered.

MedGemma is different. It is a general model that can write free text about
any image you put in front of it, but has not been approved for any clinical
use. Models like this are easy to play around with and produce fluent text
that makes it easy to forget to think if it's correct. If you're working in
digital health, you are likely to be asked to play with, buy, or oversee
tools like this. Today you will get to practise the questions you need to
ask.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from adh_python.checks import medvision as check
from adh_python.imaging import vlm
from adh_python.imaging.images import IMAGES, image_path

print("GPU available:", vlm.gpu_available())

## Load the model

`load_model()` downloads the model the first time (± 8.6 GB) then loads it
to the GPU. This takes a few minutes to run and is done when the cell prints
"Model ready."

Without a GPU, `load_model()` explains this and returns `None` instead of a
model. You read this in Level 8. The function also does this if it fails to
download or load the model after explaining what went wrong. This enables
you to proceed with the saved responses or reset and retry. Every later cell
calls `vlm.describe` with the `model` and shows the saved response if
`model` is `None`.

The text of this lab sometimes refers to 'the responses captured for this
course'. These are what you see if you do not have access to a GPU. If you
do have a GPU, some cells print the recorded response after the live
response for you to compare. You can print any of these responses using:
`vlm.describe(None, image_name, prompt)`.

In [ ]:
model = vlm.load_model()
if model is None:
    print("\nThis notebook will show saved replies.")
    print(vlm.saved_reports()["provenance"]["label"])

## First read: a normal film

Start with a single film. This small function shows a bundled X-ray with
the clinical details recorded for it.

In [ ]:
def show(image_name):
    picture = Image.open(image_path(image_name))
    plt.figure(figsize=(5, 5))
    plt.imshow(picture, cmap="gray")
    plt.title(image_name)
    plt.axis("off")
    plt.show()
    print("Clinical details:", IMAGES[image_name]["patient"])


show("normal")

The prompt is constructed using `vlm.build_prompt` which is the package's
variant of the function you wrote in Level 8 using only the minimal clinical
details. The `vlm.describe` call then prints the response as it is being
written by the model.

In [ ]:
prompt = vlm.build_prompt(IMAGES["normal"]["patient"], vlm.FOCUS, vlm.QUESTION)
print(prompt)
print("\n--- Model reply ---")
reply = vlm.describe(model, "normal", prompt)

Now compare the reply with the reference report. Each image in the course
includes the findings as described by the doctor that submitted it. Consider
this the reference for the lab but not necessarily ground truth.

In [ ]:
print("Reference findings:", IMAGES["normal"]["findings"])
if model is not None:
    print("\nFor comparison, the reply recorded for this course:\n")
    vlm.describe(None, "normal", prompt)

Did the clinical details change what the model looked at? The history of a
collision in the reply recorded for this course seems to have made it
include that there wasn't a fracture. Context guides attention, this is
useful when the context is accurate and problematic when it is not.

## First read: lobar pneumonia

Same prompt but with the patient's details included on a film with a clear
abnormality.

In [ ]:
show("lobar_pneumonia")
prompt = vlm.build_prompt(IMAGES["lobar_pneumonia"]["patient"], vlm.FOCUS, vlm.QUESTION)
print(prompt)
print("\n--- Model reply ---")
reply = vlm.describe(model, "lobar_pneumonia", prompt)
print("\nReference findings:", IMAGES["lobar_pneumonia"]["findings"])

Did the model find the abnormality, and did it put it in the right place?

Be careful with the language of place. A frontal film is divided into
**zones** (upper, middle, and lower) and not **lobes** of the lung. The
right middle lobe projects over the right middle and lower zones and is
adjacent to the right heart border. The loss of definition of this border is
the classical finding of pathology in the middle lobe. As such a 'right
lower zone opacity' may be consistent with a middle lobe pneumonia, however,
a 'right lower lobe' refers to a different lobe.

This is hard because it is difficult to evaluate a fluent paragraph with a
one-line reference. This is the core challenge of evaluating these models
and will be encountered again in the evaluation below.

## Prompt crafting: one change at a time

Small changes to a prompt can change the reply a lot. To learn what each
change does, change **one thing at a time** like you would change one dose a
time. `vlm.PROMPTS` has four prompts that build on each other:

| Prompt | What it adds |
|--------|--------------|
| `describe` | nothing: "Describe this chest X-ray." |
| `focused` | what to look at, and a single question |
| `persona` | a role for the model: a consultant radiologist |
| `structured` | a fixed format: Findings, Impression, Confidence |

Run all four on the pneumonia film. This makes four model runs so takes a
minute or two on a GPU.

In [ ]:
for prompt_id in ["describe", "focused", "persona", "structured"]:
    print(f"=== {prompt_id} ===")
    print("Prompt:", vlm.PROMPTS[prompt_id])
    print("--- Model reply ---")
    vlm.describe(model, "lobar_pneumonia", vlm.PROMPTS[prompt_id])
    print()

Which change made the reply most useful to you and which made it longer
without being better?

**A worked comparison**, judged from the replies recorded for this course.
A live run will word things differently, and other judgements are fine if
you can say why.

- **Most useful: `persona`** (about 65 words). It gives the main finding
  first, consolidation in the right lower lung field, likely pneumonia,
  then the negatives a ward team needs (heart size normal, left lung
  clear, no effusion or pneumothorax), without the general disclaimer.
  "Right lower lung field" is a zone, which fits the reference's right
  middle lobe pneumonia.
- **Longer without being better: `focused`** (about 120 words, against
  about 75 for `describe`). Its main finding is the same as in
  `describe`, an opacity in the right lower lung field. The extra length is
  a bulleted breakdown restating the same negatives, a line saying the
  ribs and clavicles are intact, and a caveat.
- **Shortest but not safest: `structured`** (± 40 words). This was the
  easiest to scan but the impression swaps 'right lower lung field' to
  'right lower lobe consolidation' which is a different lobe to the
  reference and is given with high confidence. The fixed format made the
  response more certain but not more accurate.

A change that benefits one image can harm another. Run the `persona` prompt
on both **normal** films. In the responses recorded for this course, the
exact prompt resulted in the model claiming a small opacity in the left
lower lobe of one of the normal films and a small left pleural effusion on
the other. See if you get the same.

In [ ]:
for image_name in ["normal", "normal_2"]:
    print(f"=== {image_name} ===")
    vlm.describe(model, image_name, vlm.PROMPTS["persona"])
    print("\nReference findings:", IMAGES[image_name]["findings"])
    if model is not None:
        print("\nFor comparison, the reply recorded for this course:\n")
        vlm.describe(None, image_name, vlm.PROMPTS["persona"])
    print()

The lesson isn't that personas are bad, it's that you can't judge a prompt
on one film, always test a change on normal films as well as abnormal ones.

### Your turn (optional, GPU only): write a prompt

Write your own focus and question for the heart failure film e.g., asking
about heart size or if there are any signs of fluid overload. Build the
prompt using the `vlm.build_prompt` function. Keep the clinical details
light.

This needs a live model: there is no saved reply for a prompt you write.
Without a GPU the cell below still runs, prints your prompt and says so;
do the comparison after it instead.

In [ ]:
show("heart_failure")
my_focus = "the heart size, the pulmonary vessels and the costophrenic angles"
my_question = "Are there signs of fluid overload, and if so, which?"
my_prompt = vlm.build_prompt(IMAGES["heart_failure"]["patient"], my_focus, my_question)
print(my_prompt)
if model is None:
    print("\nNo model is loaded, so there is no reply to a prompt you write.")
    print("Do the comparison of two saved prompts below instead.")
else:
    print("\n--- Model reply ---")
    reply = vlm.describe(model, "heart_failure", my_prompt)
print("\nReference findings:", IMAGES["heart_failure"]["findings"])

### Your turn, with or without a GPU: compare two saved prompts

Every prompt in the prompt-crafting table above has a saved response for the
heart failure film as well as the first read prompt with this patient's
clinical details (`"first_read"` below). Select any two and run the cell to
see a comparison between the responses as well as how they compare to the
reference findings. The most informative is to select two that are only
different by a single modification such as the `persona` and `structured`
prompts (which includes the fixed format).

In [ ]:
saved_prompts = {
    prompt_id: vlm.PROMPTS[prompt_id]
    for prompt_id in ["describe", "focused", "persona", "structured"]
}
saved_prompts["first_read"] = vlm.build_prompt(
    IMAGES["heart_failure"]["patient"], vlm.FOCUS, vlm.QUESTION
)

first_choice = "persona"  # change these to compare a different pair
second_choice = "structured"
for prompt_id in [first_choice, second_choice]:
    print(f"=== {prompt_id} ===")
    print("Prompt:", saved_prompts[prompt_id])
    print("--- Saved reply ---")
    vlm.describe(None, "heart_failure", saved_prompts[prompt_id])
    print()
print("Reference findings:", IMAGES["heart_failure"]["findings"])

Which change worked and which didn't? Write two to three sentences detailing
how the change impacted the response (longer/shorter, more/less certain,
addition/removal of findings, etc.) and if it improved the response compared
to the reference. This is not a checker, it's a judgement call.

In [ ]:
my_comparison = """
Comparing persona with structured: the fixed format made the reply about a
third as long and easier to scan, and it kept the enlarged heart and the
sternotomy wires. But it did not make the reply more accurate. It calls
pulmonary oedema, which the reference says is not obvious; where the reference
describes a minor effusion, seen in the right horizontal fissure, the persona
reply said "possibly some pleural effusions" and the structured one states
definite small effusions on both sides; and it adds right lower lobe
atelectasis, which the reference does not mention. It also rates its
confidence as high. It also drops the conclusion that these are features of
heart failure. So the format helped the reader, but it made the model more
certain than the film supports.
"""
print(my_comparison)

## Probing failure modes

A useful tool is one that can say 'no' when the answer is no. Two tests:

1. **A question about a common and important negative.** Ask if the normal
   film demonstrates a pneumothorax (air between the lung and the chest
   wall). The answer is again no.
2. **A leading question.** Asking the model to describe "the mass in the
   left upper zone" for the normal film. The correct answer is that there is
   no mass and so safe responses say so; unsafe responses describe the mass.

In [ ]:
show("normal")
for prompt_id in ["pneumothorax", "leading_mass"]:
    print(f"=== {prompt_id} ===")
    print("Prompt:", vlm.PROMPTS[prompt_id])
    print("--- Model reply ---")
    vlm.describe(model, "normal", vlm.PROMPTS[prompt_id])
    print()

Two ideas from patient safety inform why this is important.

- **Anchoring.** The practice of fixating on the first idea presented to
  you. Asking a leading question anchors the model, and a confident response
  anchors the reader in turn. Providing relevant clinical details is useful
  for a reader, and a request for an X-ray must include sufficient details
  to justify the exposure (IR(ME)R 2017); however, irrelevant or incorrect
  details can anchor a reader instead. The prompts here are generally sparse
  on clinical details, mainly for data protection (see Level 8).
- **Automation bias.** Trusting the output of a computer over your own
  judgement, either by accepting a finding it makes up or missing one it
  doesn't mention. This is most profound when the tool is often correct and
  the user is time-constrained.

### Record what you saw

Record what **your** run demonstrated as `True` or `False`. If you used the
recorded replies, you need to judge that. The checker checks the form of
answer and not the answer itself.

In [ ]:
# Judged from the recorded replies; a live run may differ.
failure_notes = {
    "invented_pneumothorax": False,
    "invented_mass": True,
}
check.failure_modes(failure_notes)

## A small evaluation

A fair test is where you ask every case the **same** question. In this case
you run the `structured` prompt on all five included films and record if the
model's main finding matched the reference. The `structured` prompt has no
clinical details so each film receives exactly the same text. This makes
five model runs.

**Decide what "agrees" means now, before you see any reply**, and apply it
the same way to every film. For example: an abnormal film agrees if the
model names the right kind of abnormality on the correct side (left or
right); a normal film agrees if the model reports no abnormality. Write
your rule down.

Responses are cut off after 400 tokens (`vlm.MAX_NEW_TOKENS`) so if the
response is long it may be cut off mid-sentence. Adjudicate based on what is
present.

In [ ]:
replies = {}
for image_name in IMAGES:
    show(image_name)
    print("--- Model reply ---")
    replies[image_name] = vlm.describe(model, image_name, vlm.PROMPTS["structured"])
    print("\nReference findings:", IMAGES[image_name]["findings"])
    print("\n")

On a GPU this cell prints the replies recorded for this course for the same
prompt so you can compare against your live run. Without a GPU you have just
seen these so it prints nothing.

In [ ]:
if model is not None:
    for image_name in IMAGES:
        print(f"=== {image_name} ===")
        vlm.describe(None, image_name, vlm.PROMPTS["structured"])
        print()

### Record your judgement

Fill in one dict per image with the `model_finding` being a few words
indicating what the model indicated the main finding was and the `agrees`
being `True` if this aligns with the main finding of the reference according
to the rule you wrote down, or `False` if not.

In [ ]:
# Judged from the recorded replies with the rule above; a live run may differ.
results = [
    {"image": "normal", "model_finding": "normal chest X-ray", "agrees": True},
    {"image": "normal_2", "model_finding": "normal chest X-ray", "agrees": True},
    {
        # Side and consolidation agree, but the impression names the
        # right lower lobe, not the middle lobe: a stricter rule says False.
        "image": "lobar_pneumonia",
        "model_finding": "right lower lobe consolidation",
        "agrees": True,
    },
    {
        "image": "multifocal_consolidation",
        "model_finding": "bilateral patchy opacities, mainly right upper and mid zones",
        "agrees": True,
    },
    {
        # The main finding (an enlarged heart, read as heart failure) agrees.
        # But it calls pulmonary oedema, which the reference says is not
        # obvious; reports effusions on both sides where the reference
        # describes a minor effusion seen in the right horizontal fissure;
        # and adds right lower lobe atelectasis, which the reference omits.
        "image": "heart_failure",
        "model_finding": "cardiomegaly, pulmonary oedema, small effusions",
        "agrees": True,
    },
]
check.evaluation_table(results)

### Count the agreement

Use a loop to count the rows where `agrees` is `True`, store the count in
`n_agree`, then work out `percent`, the percentage of films where the model
agreed with the reference.

In [ ]:
n_agree = 0
for row in results:
    if row["agrees"]:
        n_agree += 1
percent = 100 * n_agree / len(results)
print(f"The model agreed with the reference on {n_agree} of {len(results)} films.")
check.agreement(results, n_agree, percent)

### What five films can and cannot tell you

In the recorded replies for this course the model agreed with the reference
for all five films according to the example rule above. Even 5 out of 5 is
consistent with a true agreement rate anywhere from about 48% to 100% (the
95% confidence interval), and 4 out of 5 is consistent with anywhere from
about 28% to over 99%. Five films are insufficient to differentiate a useful
tool from a dangerous one.

There are other issues that can also be a problem:

- **The cases are not representative.** Five clear teaching films are not
  a week of portable films from a medical admissions unit.
- **The model may have seen them before.** `normal_2` is the exact image
  used in the example on MedGemma's own model card and all five are public.
  A model can do well on images it was trained near.
- **Your judgement is the measurement.** Two readers can score the same
  response differently. In reality, evaluations use a rubric and multiple
  readers.
- **Stated confidence is not evidence.** In the recorded replies the model
  said its confidence was high on every film, including the one where it
  named the wrong lobe.
- **Fluent can mean formulaic.** The recorded structured replies for the
  two normal films are word for word the same: a template "normal" report,
  perhaps learned by heart, rather than a description of each film. And in
  the recorded `describe` reply for the first normal film the model read
  the side markers "Dx" and "Sin" (Latin for right and left) as possibly a
  date: a small, confident invention.
- **Agreement is not safety.** One invented mass can matter more than ten
  correct normals, so count the kinds of errors, not just the total.

## Reflection: what would you need to see?

Imagine a supplier offered your department a tool like this to draft chest
X-ray reports, what evidence would you want to see before it was used on
patients?

Some ideas to get you started:

- **Intended use.** What exactly is it for, who reads its output, and what
  happens when it is wrong?
- **Evaluation on local patients.** Performance measured on data like
  yours, ideally prospectively, against the standard it would replace, and
  reported separately for groups such as age, sex and type of film.
- **Failure analysis.** What types of error it makes, not just how often it
  makes them.
- **Regulation** (as of 25 October 2026). Whether software is a medical
  device depends on its intended purpose, as its manufacturer states it;
  software intended to inform diagnosis usually is. In Great Britain a
  medical device needs UKCA marking or, under current transition
  arrangements, CE marking, and the MHRA registers devices. The route to
  UKCA marking depends on the device's risk class: for a low-risk Class I
  device the manufacturer declares conformity itself (a UK Approved Body is
  needed only for a sterile or measuring function), while higher classes
  need assessment by a UK Approved Body. On 21 October 2026 the government
  published its response to a consultation and said it intends to
  legislate to recognise devices CE marked under the EU Medical Devices
  Regulation (MDR) and In Vitro Diagnostic Medical Devices Regulation
  (IVDR) indefinitely. Until that legislation is made, the current
  transition dates apply: 30 June 2028 or 30 June 2030, depending on the
  device. Northern Ireland needs CE or CE UKNI marking.
- **Clinical safety.** The clinical safety standards for England, DCB0129
  (manufacturer) and DCB0160 (deploying organisation), are mandatory. These
  have been reviewed as of 2026 by NHS England but are still in use. NHS
  England's Digital Technology Assessment Criteria (DTAC) is also expected
  by NHS organisations.
- **Data protection.** A Data Protection Impact Assessment (UK GDPR Article
  35), and sign off from the Caldicott Guardian and information governance
  team.
- **Monitoring.** How performance is checked after go-live, who pulls the
  plug if it starts to drift, manufacturer's post-market surveillance and
  reporting of issues to the MHRA via the Yellow Card scheme.

Write your answer in the below cell. There is no checker for this, it is a
judgement call.

In [ ]:
my_reflection = """
A clear intended use (for example, drafting reports that a radiologist always
checks) and a prospective evaluation on our own patients, with error types and
results by subgroup reported. Evidence that it is a medical device with UKCA
marking, or CE marking under the current transition arrangements, after the
conformity assessment its risk class requires. DCB0129 and DCB0160 safety
cases, an assessment against NHS England's Digital Technology Assessment
Criteria (DTAC), and a Data Protection Impact Assessment with Caldicott
Guardian and information governance sign-off. Finally, a plan for monitoring
after go-live, including Yellow Card reporting of problems.
"""
print(my_reflection)

## Before you finish

- A vision language model writes fluent text regardless of if it is correct.

Evaluate the content and not the confidence.
- Change one thing at a time and test every change on normal as well as
  abnormal films.
- Leading questions and irrelevant / incorrect clinical details can anchor
  both the model and you.
- A handful of cases do not make anything safe. Ask for evaluations on
  representative patients, error analysis, regulation, and monitoring.
- Never upload real patient images / records to Colab or to a tool that your
  organisation has not approved.